# 🤖 Cuaderno 04 — Aplicaciones

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Cuatro problemas donde hay que **decidir** entre interpolar y ajustar: trayectoria de un robot, perfil de una imagen de líneas, calibración de un sensor y optimización con una parábola.

> 📖 Fichas: [Robot](../docs/aplicaciones/ap1_trayectoria_robot.md) · [Fourier](../docs/aplicaciones/ap2_fourier_lineas_imagen.md) · [Sensor](../docs/aplicaciones/ap3_calibracion_sensor.md)

In [ ]:
#@title ⚙️ Entorno (ejecuta PRIMERO)
import os, sys, importlib

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _buscar(nombre, saltos=6):
    """Busca un archivo junto al cuaderno, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


rutas = {n: _buscar(n) for n in ("robomat.py", "ajuste_curvas.py")}
faltan = [n for n, r in rutas.items() if r is None]
if faltan:
    raise FileNotFoundError(
        "No encontré " + ", ".join(faltan) +
        ". En Colab, sube la carpeta lib/ junto al cuaderno.")

_carpeta_lib = os.path.dirname(rutas["robomat.py"])
if _carpeta_lib not in sys.path:
    sys.path.insert(0, _carpeta_lib)

import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

import ajuste_curvas as ac
importlib.reload(ac)

import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider
except ImportError:
    interact = None

print("✅ Entorno cargado desde:", _carpeta_lib)
print("✅ Entorno:", "Google Colab" if ES_COLAB else "local")

### 🎯 Objetivos

- Interpolar waypoints de un robot y estimar su velocidad.
- Reconstruir el perfil de una imagen con pocos armónicos de Fourier.
- Elegir entre modelo lineal y cuadrático con criterios cuantitativos.
- Usar la interpolación parabólica para localizar un óptimo (capítulo 4).

**Aviso de honestidad:** las tres primeras aplicaciones son **extensiones
propuestas**; el método sí viene del libro. La cuarta es el ejemplo 13.2 del
capítulo 4.

## 1. Robot móvil: interpolar waypoints

Los datos son exactos (los fija el operador), así que se interpola. Se ajusta
un polinomio de Newton de grado 5 a $x(t)$ y otro a $y(t)$.

In [ ]:
t = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
xr = np.array([0.0, 1.2, 2.1, 3.4, 4.8, 6.0])
yr = np.array([0.0, 0.8, 1.9, 2.0, 1.4, 0.6])
VMAX = 2.0

tt = np.linspace(t.min(), t.max(), 400)
px = ac.newton(t, xr, tt)
py = ac.newton(t, yr, tt)

plt.figure(figsize=(8, 5))
plt.plot(xr, yr, "o", ms=9, label="waypoints")
plt.plot(px, py, "-", lw=2, label="trayectoria interpolada")
plt.axis("equal"); plt.grid(alpha=0.3)
plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.legend()
plt.title("Trayectoria del robot (grado 5)")
plt.show()

print("pasa por los waypoints:",
      np.allclose(ac.newton(t, xr, t), xr) and np.allclose(ac.newton(t, yr, t), yr))

In [ ]:
# Velocidad por diferencias finitas centrales
vx = np.gradient(px, tt)
vy = np.gradient(py, tt)
rapidez = np.hypot(vx, vy)

print(f"rapidez máxima = {rapidez.max():.3f} m/s")
print("¿excede el límite de", VMAX, "m/s?", rapidez.max() > VMAX)

plt.figure(figsize=(8, 4))
plt.plot(tt, rapidez)
plt.axhline(VMAX, color="r", ls="--", label="límite")
plt.xlabel("t (s)"); plt.ylabel("|v| (m/s)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Rapidez estimada")
plt.show()

**Interpretación.** Si la rapidez excede el límite, el problema no se arregla
cambiando de método: hay que **rediseñar los waypoints**. La interpolación
sólo describe lo que el operador pidió.

### Variante con datos ruidosos

Si los waypoints vienen de un GPS, interpolar reproduce el ruido. Se comparan
la interpolación y un ajuste cuadrático por mínimos cuadrados.

In [ ]:
rng = np.random.default_rng(7)
xn = xr + 0.12 * rng.standard_normal(t.size)
yn = yr + 0.12 * rng.standard_normal(t.size)

pxn = ac.newton(t, xn, tt)
pyn = ac.newton(t, yn, tt)

fx = ac.minimos_cuadrados(t, xn, 2)
fy = ac.minimos_cuadrados(t, yn, 2)
sx = ac.evaluar_polinomio(fx["coef"], tt)
sy = ac.evaluar_polinomio(fy["coef"], tt)

plt.figure(figsize=(8, 5))
plt.plot(xr, yr, "o", ms=8, label="waypoints reales")
plt.plot(xn, yn, "x", ms=9, label="waypoints con ruido")
plt.plot(pxn, pyn, ":", label="interpolación (sigue el ruido)")
plt.plot(sx, sy, "-", lw=2, label="mínimos cuadrados grado 2")
plt.axis("equal"); plt.grid(alpha=0.3); plt.legend()
plt.title("Datos ruidosos: interpolar vs ajustar")
plt.show()

print(f"sy/x en x(t): {fx['syx']:.4f} m   r2 = {fx['r2']:.4f}")
print(f"sy/x en y(t): {fy['syx']:.4f} m   r2 = {fy['r2']:.4f}")

## 2. Imagen de líneas con series de Fourier

Se genera una imagen sintética de franjas verticales, se toma la fila central
como señal 1-D y se reconstruye con $m$ armónicos (ecuaciones 19.14–19.16).

In [ ]:
N = 128
periodo = 16
xx, yy = np.meshgrid(np.arange(N), np.arange(N))
ideal = 0.5 + 0.5 * np.sign(np.sin(2 * np.pi * xx / periodo))
gradiente = 0.15 * xx / N
rng = np.random.default_rng(1)
imagen = np.clip(ideal + gradiente + 0.05 * rng.standard_normal((N, N)), 0, 1)

plt.figure(figsize=(5, 5))
plt.imshow(imagen, cmap="gray", vmin=0, vmax=1)
plt.title("Imagen sintética de líneas")
plt.axis("off")
plt.show()

In [ ]:
perfil = imagen[N // 2, :]
t_pix = np.arange(N, dtype=float)

# Periodo por FFT: el pico dominante (sección 19.6 del libro)
espectro = np.abs(np.fft.rfft(perfil - perfil.mean()))
frecuencias = np.fft.rfftfreq(N, d=1.0)
k = int(np.argmax(espectro[1:]) + 1)
periodo_est = N / k
omega0 = 2 * np.pi / periodo_est
print(f"armónico dominante k = {k}, periodo estimado = {periodo_est:.2f} pixeles")

plt.figure(figsize=(9, 3.5))
plt.plot(t_pix, perfil, label="perfil medido")
plt.xlabel("pixel"); plt.ylabel("intensidad"); plt.grid(alpha=0.3)
plt.legend(); plt.title("Fila central de la imagen")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(t_pix, ideal[N // 2, :], "k--", lw=1.5, label="ideal")
plt.plot(t_pix, perfil, color="0.65", lw=1, label="medido")

for m in (1, 3, 5, 10):
    coef = ac.coeficientes_fourier(t_pix, perfil, omega0, m)
    recon = ac.evaluar_fourier(t_pix, coef, omega0)
    ecm = np.sqrt(np.mean((recon - perfil) ** 2))
    plt.plot(t_pix, recon, lw=1.5, label=f"m={m} (ECM={ecm:.3f})")

plt.xlabel("pixel"); plt.ylabel("intensidad"); plt.legend(ncol=2)
plt.grid(alpha=0.3); plt.title("Reconstrucción con m armónicos")
plt.show()

In [ ]:
# Bordes: máximos de |d perfil/dx|, comparados con los bordes ideales
from scipy.signal import find_peaks

coef10 = ac.coeficientes_fourier(t_pix, perfil, omega0, 10)
recon10 = ac.evaluar_fourier(t_pix, coef10, omega0)
derivada = np.abs(np.gradient(recon10))
picos, _ = find_peaks(derivada, height=0.25 * derivada.max(), distance=4)
bordes = t_pix[picos]

print("bordes detectados (pixeles):", bordes.astype(int))
print("bordes ideales (cada 8 pixeles):", np.arange(0, N, 8))
print("el borde en 0 coincide con el límite de la ventana y no se detecta")

**Lectura.** Con $m=1$ se ve la onda fundamental; con $m=3$ y $m=5$ aparecen
los flancos; con $m=10$ el perfil es casi idéntico. El ECM baja al subir $m$,
pero llega un punto en que se empieza a copiar el ruido. Es la misma
disyuntiva que el grado del polinomio en el cuaderno 03.

## 3. Calibración de un sensor: lineal vs cuadrático

El sensor tiene ruido de $\pm0.2$ °C. Se comparan ambos modelos y se prueba su
estabilidad dejando fuera un punto.

In [ ]:
xs = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90], dtype=float)
ys = np.array([0.2, 0.9, 1.8, 2.5, 3.4, 4.2, 5.4, 6.2, 7.5, 8.4])
RUIDO = 0.1

for m in (1, 2):
    r = ac.minimos_cuadrados(xs, ys, m)
    print(f"grado {m}: coef = {np.round(r['coef'], 6)}")
    print(f"          syx = {r['syx']:.4f}   r2 = {r['r2']:.5f}")

print(f"\nruido del sensor = {RUIDO} °C")
print("un syx muy por debajo del ruido indica sobreajuste")

In [ ]:
def estabilidad(m):
    cambios = []
    for i in range(len(xs)):
        keep = np.arange(len(xs)) != i
        r = ac.minimos_cuadrados(xs[keep], ys[keep], m)
        cambios.append(np.max(np.abs(r["coef"] - ac.minimos_cuadrados(xs, ys, m)["coef"])))
    return max(cambios)

for m in (1, 2):
    print(f"grado {m}: cambio máximo de coeficientes al quitar un punto = {estabilidad(m):.5f}")

xx = np.linspace(-5, 95, 200)
plt.figure(figsize=(8, 4.5))
plt.plot(xs, ys, "o", ms=8, label="datos")
for m, estilo in [(1, "--"), (2, "-")]:
    r = ac.minimos_cuadrados(xs, ys, m)
    plt.plot(xx, ac.evaluar_polinomio(r["coef"], xx), estilo,
             label=f"grado {m} (syx={r['syx']:.3f})")
plt.xlabel("cuentas"); plt.ylabel("°C"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Calibración del sensor")
plt.show()

**Decisión esperada.** Se elige el modelo **más simple** cuyo $s_{y/x}$ sea
comparable al ruido y cuyos coeficientes sean estables. En estos datos el
lineal deja $s_{y/x}=0.216$ °C, por encima del ruido de $0.1$ °C; el cuadrático
lo baja a $0.102$ °C y su término $x^2$ es estable al quitar un punto. Por eso
se justifica el modelo cuadrático. Si el término $x^2$ no redujera el error de
forma significativa, no se justificaría.

## 4. Optimización con interpolación parabólica (capítulo 4)

La misma parábola de tres puntos sirve para localizar un óptimo. Con

$$f(x)=2\sin x-\frac{x^2}{10},$$

se aplica la ecuación (13.7) y se descarta el extremo opuesto al nuevo punto.

In [ ]:
def fopt(x):
    return 2 * np.sin(x) - x ** 2 / 10

res = ac.interpolacion_parabolica(fopt, 0.0, 1.0, 4.0, tol=1e-8)
print(f"x óptimo ≈ {res['x_opt']:.6f}")
print(f"f óptimo ≈ {res['f_opt']:.6f}")
print(f"iteraciones = {res['iteraciones']}")
for i, (xi, fi) in enumerate(res["historial"], start=1):
    print(f"  iter {i}: x = {xi:.6f}, f = {fi:.6f}")

from scipy.optimize import minimize_scalar
ref = minimize_scalar(lambda z: -fopt(z), bounds=(0, 4), method="bounded")
print(f"\nreferencia scipy: x = {ref.x:.6f}, f = {fopt(ref.x):.6f}")
print(f"error en x: {abs(res['x_opt'] - ref.x):.2e}")

In [ ]:
xx = np.linspace(-0.5, 4.5, 300)
plt.figure(figsize=(8, 4.5))
plt.plot(xx, fopt(xx), label="$2\\sin x - x^2/10$")
plt.plot(res["x_opt"], res["f_opt"], "o", ms=9, label="óptimo parabólico")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Interpolación parabólica para optimizar")
plt.show()

## 5. Autoevaluación integradora

**Pregunta 1.** En el robot, ¿por qué la variante con ruido usa mínimos
cuadrados y no interpolación? Responde en una o dos frases.

In [ ]:
respuesta_1 = '''
Escribe aquí tu respuesta.
'''
print(respuesta_1)

**Pregunta 2.** En la imagen, el ECM baja de $m=1$ a $m=10$. ¿Significa que
$m=20$ será mejor? Justifica.

In [ ]:
respuesta_2 = '''
Escribe aquí tu respuesta.
'''
print(respuesta_2)

**Pregunta 3.** El sensor tiene ruido de $0.2$ °C y el modelo cuadrático
reporta $s_{y/x}=0.15$. ¿Qué conclusión práctica sacas?

In [ ]:
respuesta_3 = '''
Escribe aquí tu respuesta.
'''
print(respuesta_3)

## 6. Cierre de la unidad

| Aplicación | Decisión | Herramienta |
|---|---|---|
| Robot con datos exactos | Interpolar | Newton |
| Robot con ruido | Ajustar | Mínimos cuadrados |
| Imagen de líneas | Ajustar con base trigonométrica | Fourier |
| Sensor | Elegir el modelo más simple defendible | Mínimos cuadrados + $s_{y/x}$ |
| Óptimo de una función | Interpolar una parábola y buscar su vértice | (13.7) |

La pregunta que ordena todo es siempre la misma: **¿los datos son exactos o
tienen ruido, y qué se quiere hacer con el modelo?**